In [3]:
import pandas as pd
import json
import ast
import re

# Read data
df = pd.read_csv('../Data_collection/merged_movies_first200.xls', encoding='utf-8')
print(f"Total movies: {len(df)}")

# Function to parse JSON
def parse_json(x):
    if pd.isna(x) or x == '[]':
        return []
    try:
        return ast.literal_eval(x)
    except:
        try:
            return json.loads(x)
        except:
            return []

# Function to clean text for URI
def clean_uri(text):
    """Convert text to valid URI format"""
    if pd.isna(text) or not text:
        return ""## empty number return an empty value directly
    text = str(text).replace(' ', '_').replace('-', '_').replace('.', '')##convert ' ','-','.'。。。。all special characters to underscores
    text = text.replace('(', '').replace(')', '').replace("'", "")## Delete '(',   ')',   "'"
    text = text.replace('&', 'and').replace(',', '').replace(':', '')##convert "&"to"and",delete','and ':'
    text = re.sub(r'[^a-zA-Z0-9_]', '', text)## final cleanning
    return text

# Create index file (Task 2.2)
with open('movie_index.txt', 'w', encoding='utf-8') as f:     ##Create the movie_index.txt file in write mode ('w').('w') is write mode.
    for i in range(len(df)):
        row = df.iloc[i]##Iterate(遍历) over each row in a DataFrame
        
        # Movie ID
        movie_id = row.get('id', row.get('movie_id', i))
        f.write(f"ID: {movie_id}\n")##Get id column ,if not has ,use index rows,but each row has.
        
        # Title
        title = row.get('title_x', row.get('title', 'Unknown'))
        f.write(f"Title: {title}\n")##Give priority to using the title_x column.
        
        ##  Year
        release = row.get('release_date', '')
        if pd.notna(release):
            year = str(release)[:4]## Since the first 4 numbers are years of "release_date"
            f.write(f"Year: {year}\n")
        
        ##  Rating
        rating = row.get('vote_average', '')
        if pd.notna(rating):
            f.write(f"Rating: {rating}\n")##Just to get average of vote
        
        ##  Genres (for :hasGenre and :GenreName)
        genres = parse_json(row.get('genres', '[]'))##parse_json parse the genre filed .
        genre_names = [g.get('name', '') for g in genres if isinstance(g, dict)]## Extracts all Genre's name 
        if genre_names:
            f.write(f"Genres: {', '.join(genre_names)}\n")## 
            # Add Genre URIs for RDF conversion
            genre_uris = [clean_uri(g) for g in genre_names]
            f.write(f"GenreURIs: {', '.join(genre_uris)}\n")
        
        ##  Directors (for :directedBy)
        crew = parse_json(row.get('crew', '[]'))
        directors = []
        for c in crew:
            if isinstance(c, dict) and c.get('job') == 'Director':
                directors.append(c.get('name', ''))## filter the job “Director” in crew
        if directors:
            f.write(f"Director: {', '.join(directors)}\n")## allow multip directors
            director_uris = [clean_uri(d) for d in directors]
            f.write(f"DirectorURIs: {', '.join(director_uris)}\n")
        
        # Actors (for :hasActor)
        cast = parse_json(row.get('cast', '[]'))
        actors = []
        for c in cast[:5]:  # top 5 actors
            if isinstance(c, dict):
                name = c.get('name', '')## remove the empty value
                if name:
                    actors.append(name)
        if actors:
            f.write(f"Actors: {', '.join(actors)}\n")
            actor_uris = [clean_uri(a) for a in actors]
            f.write(f"ActorURIs: {', '.join(actor_uris)}\n")
        
        # Production Companies (for :producedBy and :CompanyName)
        companies = parse_json(row.get('production_companies', '[]'))
        company_names = [c.get('name', '') for c in companies if isinstance(c, dict)]## parse and get company name 
        if company_names:
            f.write(f"Companies: {', '.join(company_names[:3])}\n")## display top 3 company .
            # Add Company URIs for :CompanyName
            company_uris = [clean_uri(c) for c in company_names[:3]]
            f.write(f"CompanyURIs: {', '.join(company_uris)}\n")
        
        # Production Countries (for :producedIn)
        countries = parse_json(row.get('production_countries', '[]'))
        country_names = [c.get('name', '') for c in countries if isinstance(c, dict)]
        if country_names:
            f.write(f"Countries: {', '.join(country_names)}\n")## allow multiple country to product 
            country_uris = [clean_uri(c) for c in country_names]
            f.write(f"CountryURIs: {', '.join(country_uris)}\n")
        
        # Plot (for :plot)
        overview = row.get('overview', '')
        if pd.notna(overview) and overview:
            words = str(overview).split()## get all plot but just top 100 words.
            first_100 = ' '.join(words[:100])
            f.write(f"Plot: {first_100}\n")
        
        f.write("-" * 40 + "\n\n")## Delimiter for each record
        
        # Progress
        if (i+1) % 50 == 0:
            print(f"Processed {i+1} movies")## "i" is a counter that it from 0 to 200,so need set {i+1}

print("Index file saved as: movie_index.txt")

Total movies: 200
Processed 50 movies
Processed 100 movies
Processed 150 movies
Processed 200 movies
Index file saved as: movie_index.txt
